In [1]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# 1. Load environment variables from .env
load_dotenv()

# Verify presence of required keys
required_keys = ["GEMINI_API_KEY", "GROQ_API_KEY", "TAVILY_API_KEY"]
for key in required_keys:
    status = "Loaded" if os.getenv(key) else "MISSING"
    print(f"{key}: {status}")

GEMINI_API_KEY: Loaded
GROQ_API_KEY: Loaded
TAVILY_API_KEY: Loaded


# Document Loading & PDF Parsing

Concept: LangChain document loaders standardize raw data sources (PDFs, HTML, Markdown) into Document objects containing page_content (string) and metadata (dict). We use PyPDFLoader to parse local PDFs into clean string context.

In [2]:
file_path = './asdasd.docx'

if file_path.endswith('pdf'):
    print('pdf')
elif file_path.endswith(('.doc', 'docx')):
    print('doccccc')
else:
    print('NONE')


doccccc


In [12]:
import re

def load_resume_text(file_path: str) -> str:
    """Parses a multi-page PDF resume or docx resume into a consolidated text string."""

    if file_path.endswith('.pdf'):
        from langchain_community.document_loaders import PyMuPDFLoader

        loader = PyMuPDFLoader(file_path)
        pages = loader.load()

        raw_text = "\n\n".join([page.page_content for page in pages])

        # Clean up non-breaking spaces and white-space gaps
        resume_context = re.sub(r"\xa0", " ", raw_text)

        return resume_context
    elif file_path.endswith(('.doc', 'docx')):
        from langchain_community.document_loaders import UnstructuredWordDocumentLoader

        loader = UnstructuredWordDocumentLoader(file_path)
        pages = loader.load()

        raw_text = "\n\n".join([page.page_content for page in pages])

        # Clean up non-breaking spaces and redundant whitespace
        resume_context = re.sub(r"\xa0", " ", raw_text)
        resume_context = re.sub(r"\n\s*\n", "\n\n", resume_context)

        return resume_context
    
    else:
        pass




In [13]:
# Example usage:
resume_context = load_resume_text("./resume.pdf")

print(f"Parsed Resume Length: {len(resume_context)} characters")
print("-"*100)
print(resume_context)

Parsed Resume Length: 6334 characters
----------------------------------------------------------------------------------------------------
Hamza Ahmed Siddiqui 
Data Scientist & ML Engineer · Computer Vision & NLP · AI Solutions 
hamzaahmedsiddiqui10@gmail.com | +49 1575 5794618 | Göttingen, Germany | LinkedIn | GitHub 
 
WoRk ExPERIEncE 
~Research Assistant - Computer Vision                                                                                           Apr 2026 – Present 
Eckerlab                                                                                                                                                Göttingen, Germany 
Project: Video Generalization Dataset 
•​ Investigated cross-domain generalization in video action recognition on Something-Something V2 (SSv2) by 
benchmarking feature representations from self-supervised vision models (DINOv2, V-JEPA Base/Giant) with 
trained attention heads.    
•​ Integrated Segment Anything Model (SAM 3) into the vi

# Tool Integration with Tavily Search API
Concept: An agent's tool acts as an external context retriever. To avoid scraping blocks or enterprise OAuth setup, we query search engines with restricted domain filters site:[linkedin.com/jobs](https://linkedin.com/jobs/view) to pull live job descriptions directly.

In [ ]:
def clean_linkedin_jd(raw_text: str) -> str:
    """Extracts core job description content while discarding LinkedIn UI boilerplate."""
    
    # 1. Truncate trailing boilerplate (similar jobs, footer links, recommendations)
    cutoff_patterns = [
        r"##\s*Similar jobs",
        r"##\s*People also viewed",
        r"Get notified about new",
        r"Sign in to set job alerts",
        r"Explore top content on LinkedIn",
        r"LinkedIn\s*©\s*\d{4}"
    ]
    for pattern in cutoff_patterns:
        match = re.search(pattern, raw_text, flags=re.IGNORECASE)
        if match:
            raw_text = raw_text[:match.start()]

    # 2. Strip standard sign-in modals & banner boilerplate at the top
    clean_text = re.sub(
        r"(##\s*Sign in to access AI-powered advices|By clicking Continue to join or sign in|Select Accept to consent or Reject).*?\n\n",
        "",
        raw_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    # 3. Collapse multiple consecutive empty lines
    clean_text = re.sub(r"\n\s*\n", "\n\n", clean_text).strip()
    return clean_text

def fetch_target_jobs(role_title: str, location: str = "Germany", max_results: int = 3) -> str:
    """Fetches and extracts clean, focused LinkedIn job descriptions."""
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    
    # Restrict strictly to individual LinkedIn job posting URLs
    search_query = f'"{role_title}" {location} site:linkedin.com/jobs/view/'
    
    response = client.search(
        query=search_query, 
        max_results=max_results, 
        search_depth="advanced",
        time_range="week",
        include_raw_content=True
    )
    
    job_snippets = []
    for i, result in enumerate(response.get("results", []), 1):
        raw_content = result.get("raw_content") or result.get("content", "")
        
        # Clean the extracted page content
        extracted_jd = clean_linkedin_jd(raw_content)
        
        snippet = (
            f"=== JOB POSTING {i}: {result.get('title')} ===\n"
            f"URL: {result.get('url')}\n\n"
            f"--- CLEANED JOB DESCRIPTION ---\n"
            f"{extracted_jd}\n"
            f"{'=' * 60}\n"
        )
        job_snippets.append(snippet)
        
    return "\n\n".join(job_snippets)

In [ ]:
# Example execution:
job_data = fetch_target_jobs(role_title="Data Scientist", location="Germany", max_results=3)


In [20]:
print(job_data)

=== JOB POSTING 1: NEXUS Academy hiring Data Scientist / Senior Data Scientist in Germany | LinkedIn ===
URL: https://pk.linkedin.com/jobs/view/healthcare-specialist-at-nexus-academy-4476420084

--- CLEANED JOB DESCRIPTION ---
### Data Scientist / Senior Data Scientist

[NEXUS Academy](https://www.linkedin.com/company/nexus-bd?trk=public_jobs_sub-nav-cta-optional-url "NEXUS Academy") Germany

# Data Scientist / Senior Data Scientist

#### [NEXUS Academy](https://www.linkedin.com/company/nexus-bd?trk=public_jobs_topcard-org-name) Germany 3 days ago [See who NEXUS Academy has hired for this role](https://www.linkedin.com/login?session_redirect=https%3A%2F%2Fwww%2Elinkedin%2Ecom%2Fsearch%2Fresults%2Fpeople%2F%3FfacetCurrentCompany%3D86410713%26title%3DSenior%2BData%2BScientist&emailAddress=&fromSignIn=&trk=public_jobs_see-who-was-hired_people-search-link_face-pile-cta)

* [Report this job](https://www.linkedin.com/uas/login?fromSignIn=true&session_redirect=https%3A%2F%2Fde.linkedin.com%2F

In [38]:
from typing import List

def remove_matching_lines(text: str, remove_patterns: List[str], case_sensitive: bool = True) -> str:
    """
    Removes lines from a multi-line string if they contain any substring from remove_patterns.
    
    :param text: The input multi-line string.
    :param remove_patterns: A list of string patterns to match against each line.
    :param case_sensitive: Whether the substring matching should be case-sensitive.
    :return: A multi-line string with matching lines removed.
    """
    lines = text.splitlines()
    
    if not case_sensitive:
        patterns = [p.lower() for p in remove_patterns]
        filtered_lines = [
            line for line in lines
            if not any(pattern in line.lower() or line.lower()=='or' for pattern in patterns)
        ]
    else:
        filtered_lines = [
            line for line in lines
            if not any(pattern in line for pattern in remove_patterns)
        ]
        
    return "\n".join(filtered_lines)

blacklist = [
    "* [Report this job]",
    "Get AI-powered advice on this job and more exclusive features.",
    "or     ",
    "[Sign in](",
    "## Sign in",
    "Use AI to assess how you fit",
    "![Image",
    "New to LinkedIn? [Join now](",
    "## Sign in to evaluate your skills",
    "Referrals increase your chances of interviewing ",
    "[See who you know](",
    "Get notified when a new",
    "Clear text",
    "Direct message the job poster from"
    ]
cleaned_job_data = remove_matching_lines(job_data, blacklist, case_sensitive=False)
print(cleaned_job_data)

=== JOB POSTING 1: NEXUS Academy hiring Data Scientist / Senior Data Scientist in Germany | LinkedIn ===
URL: https://pk.linkedin.com/jobs/view/healthcare-specialist-at-nexus-academy-4476420084

--- CLEANED JOB DESCRIPTION ---
### Data Scientist / Senior Data Scientist

[NEXUS Academy](https://www.linkedin.com/company/nexus-bd?trk=public_jobs_sub-nav-cta-optional-url "NEXUS Academy") Germany

# Data Scientist / Senior Data Scientist

#### [NEXUS Academy](https://www.linkedin.com/company/nexus-bd?trk=public_jobs_topcard-org-name) Germany 3 days ago [See who NEXUS Academy has hired for this role](https://www.linkedin.com/login?session_redirect=https%3A%2F%2Fwww%2Elinkedin%2Ecom%2Fsearch%2Fresults%2Fpeople%2F%3FfacetCurrentCompany%3D86410713%26title%3DSenior%2BData%2BScientist&emailAddress=&fromSignIn=&trk=public_jobs_see-who-was-hired_people-search-link_face-pile-cta)













[Shairyar Asghar](https://pk.linkedin.com/in/shairyar-asghar) 

### Shairyar Asghar

#### Research Officer @

In [39]:
(len(job_data), len(cleaned_job_data))

(13245, 8963)